<a href="https://colab.research.google.com/github/Darklord007masai/part4-fastapi-service/blob/main/app_main_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import List
import pickle
import numpy as np
import pandas as pd
import os

app = FastAPI(
    title="D2C Customer Churn Intelligence Service",
    description="Internal API service to predict 60-day customer churn risk for CRM integration.",
    version="1.0.0"
)

# Global variables for model state
MODEL_PATH = "model.pkl"
model_artifact = None

@app.on_event("startup")
def load_model_weights():
    """Load model weights dynamically into memory during application bootstrap."""
    global model_artifact
    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(f"Critical error: Saved model weights file '{MODEL_PATH}' was not found.")
    with open(MODEL_PATH, "rb") as f:
        model_artifact = pickle.load(f)

# --- PYDANTIC INPUT DATA SCHEMAS ---
class CustomerFeatures(BaseModel):
    customer_id: str = Field(..., example="CUST-1024", description="Unique alphanumeric identifier for a customer record.")
    total_spend: float = Field(..., ge=0.0, example=249.50, description="Total dollar spend aggregated on or before the snapshot date.")
    order_count: int = Field(..., ge=0, example=5, description="Total unique order count placed on or before the snapshot date.")
    ticket_count: int = Field(..., ge=0, example=2, description="Total customer support tickets opened on or before the snapshot date.")
    web_sessions: int = Field(..., ge=0, example=18, description="Total user session interactions logged on or before the snapshot date.")

class SinglePredictionResponse(BaseModel):
    customer_id: str
    churn_probability: float
    predicted_class: int
    risk_explanation: str

class BatchPredictionResponse(BaseModel):
    predictions: List[SinglePredictionResponse]

# --- UTILITY EXPLANATION LOGIC ---
def generate_risk_explanation(probability: float, features: CustomerFeatures) -> str:
    """Generates an intuitive, readable explanation of risk factors based on inputs."""
    if probability < 0.35:
        return "Low customer risk profile. Maintain baseline product and marketing engagement."

    triggers = []
    if features.ticket_count >= 3:
        triggers.append("high support request friction")
    if features.order_count <= 1:
        triggers.append("weak historic operational retention footprint")
    if features.web_sessions < 5:
        triggers.append("declining application session footprints")

    trigger_text = " and ".join(triggers) if triggers else "general behavior fluctuations"
    return f"High risk profile triggered primarily by {trigger_text}."

# --- API ENDPOINTS ---
@app.get("/health", status_code=status.HTTP_200_OK, summary="API Health Check Status Indicator")
def health_check():
    """System heartbeat endpoint verification."""
    if model_artifact is None:
        return {"status": "unhealthy", "model_loaded": False}
    return {"status": "healthy", "model_loaded": True}

@app.post("/predict", response_model=SinglePredictionResponse, status_code=status.HTTP_200_OK, summary="Score Single Customer")
def predict_single(payload: CustomerFeatures):
    """Inference evaluation score for one customer record."""
    try:
        # Convert Pydantic payload to dataframe structure matching pipeline feature orders
        input_data = pd.DataFrame([{
            'total_spend': payload.total_spend,
            'order_count': payload.order_count,
            'ticket_count': payload.ticket_count,
            'web_sessions': payload.web_sessions
        }])

        # Extract model component
        model = model_artifact['model']

        # Calculate inference score
        prob = float(model.predict_proba(input_data)[:, 1][0])
        pred_class = int(prob >= 0.35)  # Matches project threshold metrics
        explanation = generate_risk_explanation(prob, payload)

        return SinglePredictionResponse(
            customer_id=payload.customer_id,
            churn_probability=round(prob, 4),
            predicted_class=pred_class,
            risk_explanation=explanation
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference execution engine exception: {str(e)}")

@app.post("/batch_predict", response_model=BatchPredictionResponse, status_code=status.HTTP_200_OK, summary="Score Multiple Customers")
def predict_batch(payload: List[CustomerFeatures]):
    """Bulk validation scoring interface supporting automated batch pipeline operations."""
    if len(payload) == 0:
        raise HTTPException(status_code=400, detail="Payload collection list must contain data.")

    results = []
    for item in payload:
        results.append(predict_single(item))
    return BatchPredictionResponse(predictions=results)

/tmp/ipykernel_2869/1662333936.py:19: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
/tmp/ipykernel_2869/1662333936.py:30: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  customer_id: str = Field(..., example="CUST-1024", description="Unique alphanumeric identifier for a customer record.")
/tmp/ipykernel_2869/1662333936.py:31: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydan